# 📊 Model Evaluation Metrics — Complete Guide
**Goal:** Understand every major ML evaluation metric with code examples.

> Amazon ML School expects you to deeply understand *why* we use different metrics — not just how to calculate them. This notebook demonstrates each metric with a real example.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    mean_squared_error, mean_absolute_error, r2_score
)

# Load breast cancer dataset (binary classification)
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target  # 0=malignant, 1=benign

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print("Dataset:", data.dataset_filename if hasattr(data,'dataset_filename') else "Breast Cancer Wisconsin")
print("Classes:", data.target_names)
print("Shape:", X.shape)


## 1. Accuracy, Precision, Recall, F1-Score

| Metric | Formula | When to use |
|--------|---------|-------------|
| **Accuracy** | (TP+TN)/(Total) | Balanced classes |
| **Precision** | TP/(TP+FP) | When false positives are costly |
| **Recall** | TP/(TP+FN) | When false negatives are costly (medical!) |
| **F1-Score** | 2×(P×R)/(P+R) | Imbalanced classes |

> **Medical example:** For cancer detection, high Recall matters most — we'd rather have false alarms than miss a real cancer (false negative).


In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_s, y_train)
y_pred = model.predict(X_test_s)
y_proba = model.predict_proba(X_test_s)[:, 1]

print("=" * 40)
print("   CLASSIFICATION METRICS")
print("=" * 40)
print(f"  Accuracy  : {accuracy_score(y_test, y_pred)*100:.2f}%")
print(f"  Precision : {precision_score(y_test, y_pred)*100:.2f}%")
print(f"  Recall    : {recall_score(y_test, y_pred)*100:.2f}%")
print(f"  F1 Score  : {f1_score(y_test, y_pred)*100:.2f}%")
print(f"  ROC-AUC   : {roc_auc_score(y_test, y_proba):.4f}")
print("=" * 40)


## 2. Confusion Matrix — Visualized

In [ ]:
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Predicted Malignant','Predicted Benign'],
            yticklabels=['Actual Malignant','Actual Benign'])
axes[0].set_title('Confusion Matrix', fontsize=13)

labels = ['True Negatives\n(Correct Malignant)', 'False Positives\n(Wrong Alarm)',
          'False Negatives\n(Missed Cancer!)', 'True Positives\n(Correct Benign)']
values = [tn, fp, fn, tp]
colors_bar = ['#2ECC71','#F39C12','#E74C3C','#3498DB']

axes[1].bar(range(4), values, color=colors_bar, edgecolor='black')
axes[1].set_xticks(range(4))
axes[1].set_xticklabels(labels, fontsize=9)
axes[1].set_title('Confusion Matrix Breakdown', fontsize=13)

plt.tight_layout()
plt.show()


## 3. ROC Curve — Compare Multiple Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree':       DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
}

plt.figure(figsize=(8, 6))

for name, m in models.items():
    m.fit(X_train_s, y_train)
    proba = m.predict_proba(X_test_s)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, lw=2, label=f'{name} (AUC={auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random (AUC=0.500)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves — Model Comparison', fontsize=13)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


## 4. Cross-Validation — More Reliable than a Single Split

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("5-Fold Cross-Validation Results:")
print("=" * 50)
for name, m in models.items():
    scores = cross_val_score(m, scaler.fit_transform(X), y, cv=cv, scoring='accuracy')
    print(f"{name:25s}: {scores.mean()*100:.2f}% ± {scores.std()*100:.2f}%")
print("=" * 50)


**Why Cross-Validation?** A single 80/20 split might get lucky or unlucky. Cross-validation gives us 5 different train/test splits and averages — much more reliable estimate of real-world performance.


## 5. Regression Metrics Reference

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression, Ridge, Lasso

housing = fetch_california_housing()
Xr = housing.data
yr = housing.target

Xr_train, Xr_test, yr_train, yr_test = train_test_split(Xr, yr, test_size=0.2, random_state=42)
scr = StandardScaler()
Xr_train_s = scr.fit_transform(Xr_train)
Xr_test_s  = scr.transform(Xr_test)

reg_models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression':  Ridge(alpha=1.0),
    'Lasso Regression':  Lasso(alpha=0.1),
}

print("Regression Model Comparison:")
print(f"{'Model':<22} {'R²':>8} {'RMSE':>8} {'MAE':>8}")
print("=" * 50)
for name, m in reg_models.items():
    m.fit(Xr_train_s, yr_train)
    yp = m.predict(Xr_test_s)
    r2   = r2_score(yr_test, yp)
    rmse = np.sqrt(mean_squared_error(yr_test, yp))
    mae  = mean_absolute_error(yr_test, yp)
    print(f"{name:<22} {r2:>8.4f} {rmse:>8.4f} {mae:>8.4f}")


## Metrics Quick Reference Card

### Classification
| Metric | Best Value | Formula | Use When |
|--------|-----------|---------|---------|
| Accuracy | 1.0 | (TP+TN)/N | Balanced classes |
| Precision | 1.0 | TP/(TP+FP) | Cost of false alarm is high |
| Recall | 1.0 | TP/(TP+FN) | Missing a positive is dangerous |
| F1 | 1.0 | 2PR/(P+R) | Imbalanced classes |
| ROC-AUC | 1.0 | Area under curve | Comparing models |

### Regression
| Metric | Best Value | Meaning |
|--------|-----------|---------|
| R² | 1.0 | % variance explained |
| RMSE | 0 | Penalizes large errors heavily |
| MAE | 0 | Average absolute error |
